<a href="https://colab.research.google.com/github/RewelliOliveira/data-mine-booking/blob/main/Exercicio_KNN_Perfis_Musicais.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercício — Classificação de Perfis Musicais com KNN

Utilize os dados do questionário **Perfil de Escuta Musical** para classificar participantes em três perfis:

- 🎧 Explorador Musical
- ❤️ Fã Fiel
- 🎯 Ouvinte Contextual

O KNN deverá ser implementado **sem utilizar uma biblioteca que já forneça o algoritmo KNN**.

Serão investigados:
- diferentes valores de **K**;
- diferentes medidas de distância;
- desempenho do classificador.

## Preparação das labels

As respostas Q1–Q16 são as features do modelo.

Os rótulos foram definidos a partir dos escores:

$$E=(Q1+Q2+Q3+Q4+Q5)/5$$

$$F=(Q6+Q7+Q8+Q9+Q10)/5$$

$$C=(Q11+Q12+Q13+Q14+Q15+Q16)/6$$

O maior escore determina o `label`.

**Importante:** não utilize a a classe (perfil escolhido) pelo participante como feature do KNN.

## 2. Carregando os dados

Coloque o arquivo `respostas_musicais.csv` no mesmo diretório do notebook.

In [ ]:
import pandas as pd
import numpy as np

dados = pd.read_csv('https://raw.githubusercontent.com/RewelliOliveira/data-mine/refs/heads/main/data/Perfil%20de%20Escuta%20Musical%20(respostas)%20-%20Respostas%20ao%20formul%C3%A1rio%201.csv')

### Perguntas

1. Quantas pessoas responderam?
2. Quantas colunas existem?
3. Existem valores ausentes?
4. Atribua o valor dos rótulos de classe segundo a regra acima. Quantas pessoas pertencem a cada classe?

**Responda aqui.**

In [ ]:
print("1. Quantas pessoas responderam?", dados.shape[0])
print("2. Quantas colunas existem?", dados.shape[1])
print("3. Quantidade total de valores ausentes:", dados.isna().sum().sum())

colunas_q = [f"Q{i}" for i in range(1, 17)]
dados.rename(columns=dict(zip(dados.columns[1:17], colunas_q)), inplace=True)

escores = pd.DataFrame({
    "Explorador Musical": dados[[f"Q{i}" for i in range(1, 6)]].mean(axis=1),
    "Fã Fiel": dados[[f"Q{i}" for i in range(6, 11)]].mean(axis=1),
    "Ouvinte Contextual": dados[[f"Q{i}" for i in range(11, 17)]].mean(axis=1)
})

dados["label"] = escores.idxmax(axis=1)

print("4. Quantidade de pessoas em cada classe:")
print(dados["label"].value_counts().reindex(escores.columns, fill_value=0))


## 3. Criando X e y

Utilize Q1–Q16 como atributos e `label` como variável de saída.

In [ ]:
X = dados[colunas_q].copy()

X = X.fillna(X.median()).to_numpy(dtype=float)
y = dados["label"].to_numpy()

print("Instâncias:", X.shape[0])
print("Atributos:", X.shape[1])


## 4. Separação entre treinamento e validação

Utilize 70% dos dados para treinamento e 30% para validação. Faça a divisão utilizando apenas NumPy.

In [ ]:
np.random.seed(42)

indices = np.random.permutation(len(X))
n_validacao = int(0.30 * len(X))

indices_validacao = indices[:n_validacao]
indices_treino = indices[n_validacao:]

X_train = X[indices_treino]
y_train = y[indices_treino]
X_validacao = X[indices_validacao]
y_validacao = y[indices_validacao]

print("Treinamento:", len(X_train))
print("Validação:", len(X_validacao))


### Respostas

1. O conjunto de teste deve representar dados novos. Se ele for usado para ajustar o treinamento, a avaliação fica tendenciosa e pode parecer melhor do que realmente é.

2. É importante ter exemplos de todas as classes para que o modelo consiga aprendê-las e para que a avaliação seja justa. Nesta base não há participantes da classe **Explorador Musical** e, com a divisão sorteada, a validação contém apenas **Fã Fiel**. Por isso, a acurácia obtida deve ser interpretada com cuidado.


## 5. Distância Euclidiana

Implemente:

$$d(x,z)=\sqrt{\sum_i(x_i-z_i)^2}$$

In [ ]:
def distancia_euclidiana(a, b):
    return np.sqrt(np.sum((a - b) ** 2))


### Teste

Compare a distância entre duas pessoas do conjunto de treinamento.

In [ ]:
distancia_euclidiana(X_train[0], X_train[1])

### Respostas

1. Uma distância pequena significa que as duas pessoas deram respostas parecidas e possuem perfis musicais semelhantes.

2. Uma distância grande indica respostas mais diferentes e, portanto, perfis musicais menos semelhantes.


## 6. Implementação do KNN

Para cada exemplo de teste:

1. calcule a distância até todos os exemplos de treinamento;
2. ordene as distâncias;
3. selecione os K vizinhos mais próximos;
4. faça uma votação majoritária.

In [ ]:
def knn_predict(X_train, y_train, X_test, k=3,
                funcao_distancia=distancia_euclidiana):

    previsoes = []

    for x in X_test:
        distancias = []

        for i in range(len(X_train)):
            d = funcao_distancia(x, X_train[i])
            distancias.append((d, y_train[i]))

        distancias.sort(key=lambda item: item[0])
        vizinhos = distancias[:k]

        classes = [classe for _, classe in vizinhos]
        classes_unicas, contagens = np.unique(classes, return_counts=True)

        previsao = classes_unicas[np.argmax(contagens)]
        previsoes.append(previsao)

    return np.array(previsoes)

## 7. Primeiro experimento — K = 3

In [ ]:
previsoes = knn_predict(X_train, y_train, X_validacao, k=3)

corretas = np.sum(previsoes == y_validacao)
incorretas = len(y_validacao) - corretas
acuracia = corretas / len(y_validacao)

print(f"Acurácia: {acuracia:.3f}")
print("Corretas:", corretas)
print("Incorretas:", incorretas)


### Respostas

1. A acurácia foi de **0,750**, ou **75%**.
2. Foram **3 classificações corretas**.
3. Houve **1 classificação incorreta**.

Como as quatro pessoas da validação pertencem à mesma classe, esse resultado é pouco representativo para avaliar as outras classes.


## 8. Testando diferentes valores de K

Teste:

**K = 1, 3, 5, 7, 9 e 11**

Compare as acurácias.

In [ ]:
valores_k = [1, 3, 5, 7, 9, 11]
resultados_k = []

for k in valores_k:
    previsoes = knn_predict(X_train, y_train, X_validacao, k=k)
    acuracia = np.mean(previsoes == y_validacao)
    resultados_k.append(acuracia)
    print(f"K = {k}: {acuracia:.3f}")


In [ ]:
import matplotlib.pyplot as plt

plt.plot(valores_k, resultados_k, marker="o")
plt.xlabel("K")
plt.ylabel("Acurácia")
plt.title("Acurácia para diferentes valores de K")
plt.xticks(valores_k)
plt.grid(True)
plt.show()

### Respostas

1. Os valores **K = 7, 9 e 11** empataram com a melhor acurácia: **100%**.
2. Não. O menor valor, K = 1, obteve 75%.
3. O maior K empatou com os melhores neste teste, mas isso não significa que o maior valor será sempre o melhor.
4. Um K muito pequeno deixa o modelo sensível a ruídos e casos isolados.
5. Um K muito grande pode favorecer a classe majoritária e ignorar padrões das classes menores.


## 9. Outras medidas de distância

Implemente:

### Manhattan

$$d(x,z)=\sum_i |x_i-z_i|$$

### Minkowski

$$d(x,z)=\left(\sum_i|x_i-z_i|^p\right)^{1/p}$$

Utilize inicialmente $p=3$.

In [ ]:
def distancia_manhattan(a, b):
    return np.sum(np.abs(a - b))


In [ ]:
def distancia_minkowski(a, b, p=3):
    return np.sum(np.abs(a - b) ** p) ** (1 / p)


### Teste

Compare as três medidas para duas instâncias.

In [ ]:
a = X_train[0]
b = X_train[1]

print("Euclidiana:", distancia_euclidiana(a, b))
print("Manhattan:", distancia_manhattan(a, b))
print("Minkowski (p=3):", distancia_minkowski(a, b, p=3))

### Respostas

1. Não. Para as duas instâncias usadas no teste, as distâncias foram aproximadamente **9,055** (Euclidiana), **30** (Manhattan) e **6,366** (Minkowski com p = 3).
2. Elas combinam as diferenças entre os atributos de maneiras diferentes. A Manhattan soma as diferenças absolutas, enquanto Euclidiana e Minkowski dão pesos diferentes às diferenças maiores.
3. Significa mudar o critério usado para decidir quais pessoas são mais parecidas, o que pode alterar os vizinhos escolhidos e a classificação final.


## 10. Generalizando o KNN

A função `knn_predict` já permite informar a função de distância.

Exemplo:

```python
knn_predict(
    X_train, y_train, X_test,
    k=5,
    funcao_distancia=distancia_manhattan
)
```

## 11. Comparando as medidas

Utilize **K = 5** e compare:

- Euclidiana;
- Manhattan;
- Minkowski ($p=3$).

In [ ]:
funcoes = {
    "Euclidiana": distancia_euclidiana,
    "Manhattan": distancia_manhattan,
    "Minkowski (p=3)": lambda a, b: distancia_minkowski(a, b, p=3)
}

for nome, funcao in funcoes.items():
    previsoes = knn_predict(
        X_train, y_train, X_validacao,
        k=5, funcao_distancia=funcao
    )
    acuracia = np.mean(previsoes == y_validacao)
    print(f"{nome}: {acuracia:.3f}")


### Respostas

1. Minkowski apresentou a maior acurácia: **100%**. Euclidiana e Manhattan obtiveram **75%**.
2. A diferença foi de **25 pontos percentuais** nesse teste.
3. A medida de distância muda quais exemplos são considerados mais próximos. Com isso, os vizinhos e a classe prevista também podem mudar.


## 12. Experimento completo — K × distância

Teste todas as combinações:

- K = 1, 3, 5, 7, 9, 11;
- Euclidiana;
- Manhattan;
- Minkowski ($p=3$).

Construa uma tabela com as acurácias.

In [ ]:
resultados = []

for nome, funcao in funcoes.items():
    for k in valores_k:
        previsoes = knn_predict(
            X_train, y_train, X_validacao,
            k=k, funcao_distancia=funcao
        )
        acuracia = np.mean(previsoes == y_validacao)
        resultados.append([nome, k, acuracia])

tabela = pd.DataFrame(
    resultados,
    columns=["Distância", "K", "Acurácia"]
)

tabela


In [ ]:
tabela_pivot = tabela.pivot(
    index="Distância",
    columns="K",
    values="Acurácia"
)

tabela_pivot

### Respostas

1. A melhor acurácia foi **100%**. Ela ocorreu com Euclidiana e Manhattan para K = 7, 9 e 11, e com Minkowski para K = 5, 7, 9 e 11.
2. A pior acurácia foi **75%**: Euclidiana e Manhattan com K = 1, 3 ou 5, e Minkowski com K = 1 ou 3.
3. Não exatamente. O menor K com resultado máximo foi 7 para Euclidiana e Manhattan, mas 5 para Minkowski.
4. Sim. Em K = 5, por exemplo, Minkowski atingiu 100%, enquanto Euclidiana e Manhattan ficaram em 75%.
5. Não. Neste experimento a acurácia aumentou e depois ficou estável, mas valores maiores de K não garantem melhora em outras bases.

Esses resultados devem ser vistos com cautela, pois a validação possui somente quatro exemplos, todos da classe Fã Fiel.


Agora utilize os melhores parâmetros no conjunto de dados coletado na aula.

## Preparação das labels

As respostas Q1–Q16 são as features do modelo.

Os rótulos foram definidos a partir dos escores:

$$E=(Q1+Q2+Q3+Q4+Q5)/5$$

$$F=(Q6+Q7+Q8+Q9+Q10)/5$$

$$C=(Q11+Q12+Q13+Q14+Q15+Q16)/6$$

O maior escore determina o `label`. Utilize essa regra para calcular a label do conjunto de teste.

**Importante:** não utilize a a classe (perfil escolhido) pelo participante como feature do KNN.

In [ ]:
melhor_k = 5
melhor_distancia = lambda a, b: distancia_minkowski(a, b, p=3)

previsoes_finais = knn_predict(
    X_train, y_train, X_validacao,
    k=melhor_k,
    funcao_distancia=melhor_distancia
)

resultado_final = pd.DataFrame({
    "Classe real": y_validacao,
    "Classe prevista": previsoes_finais
})

print(resultado_final)
print("Acurácia final:", np.mean(previsoes_finais == y_validacao))
